In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

# Target
y = df["INDICATED_DAMAGE"]
X = df.drop(columns=["INDICATED_DAMAGE"])

# Drop high-noise columns (example)
X = X.drop(columns=["REMARKS", "COMMENTS", "INDEX_NR"])

# Convert categorical columns
cat_cols = X.select_dtypes(include="object").columns

for col in cat_cols:
    X[col] = X[col].astype("category")

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    enable_categorical=True  # important
)

model.fit(X_train, y_train)

preds = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, preds))

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

# =========================================================
# CHANGE THIS TO "random_forest" OR "xgboost"
# =========================================================
MODEL_TYPE = "xgboost"

# =========================================================
# FILE PATHS
# =========================================================
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
OUTPUT_PATH = "test_predictions.csv"

# =========================================================
# LOAD DATA
# =========================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# =========================================================
# BASIC CLEANING
# =========================================================
# Target column
TARGET = "INDICATED_DAMAGE"

# Drop columns that are IDs / leakage / too messy as raw text
drop_cols = [
    "INDEX_NR",
    "REMARKS",
    "COMMENTS",
    "BIRD_BAND_NUMBER",
    "REG",
    "FLT",
    "AIRPORT",
    "OPERATOR",
    "SPECIES",
    "INCIDENT_DATE",
    "LUPDATE"
]

# Only drop columns that actually exist
drop_cols = [col for col in drop_cols if col in train_df.columns]

# Separate target
y = train_df[TARGET]
X = train_df.drop(columns=[TARGET])
X_test = test_df.copy()

# Drop unwanted columns
X = X.drop(columns=drop_cols, errors="ignore")
X_test = X_test.drop(columns=drop_cols, errors="ignore")

# =========================================================
# FEATURE ENGINEERING
# =========================================================
def add_features(df):
    df = df.copy()

    # Convert TIME to hour
    if "TIME" in df.columns:
        df["TIME"] = df["TIME"].astype(str)
        df["HOUR"] = pd.to_datetime(df["TIME"], format="%H:%M", errors="coerce").dt.hour

    # Numeric conversions
    numeric_cols_to_force = [
        "LATITUDE", "LONGITUDE", "HEIGHT", "SPEED", "DISTANCE",
        "AC_MASS", "NUM_ENGS", "INCIDENT_MONTH", "INCIDENT_YEAR"
    ]

    for col in numeric_cols_to_force:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Binary features
    if "WARNED" in df.columns:
        df["WARNED_BIN"] = df["WARNED"].astype(str).str.strip().str.lower().map({
            "yes": 1, "no": 0
        })

    if "REMAINS_COLLECTED" in df.columns:
        df["REMAINS_COLLECTED_BIN"] = pd.to_numeric(df["REMAINS_COLLECTED"], errors="coerce")

    if "REMAINS_SENT" in df.columns:
        df["REMAINS_SENT_BIN"] = pd.to_numeric(df["REMAINS_SENT"], errors="coerce")

    # Parse approximate counts like "10-Feb", "2-10", etc.
    count_map = {
        "1": 1,
        "2-10": 6,
        "11-100": 55,
        "over 100": 150,
        "01-oct": 6,
        "10-feb": 6,
        "10-mar": 7,
        "100+": 150
    }

    for col in ["NUM_SEEN", "NUM_STRUCK"]:
        if col in df.columns:
            temp = df[col].astype(str).str.strip().str.lower()
            df[col + "_PARSED"] = pd.to_numeric(temp, errors="coerce")
            df.loc[df[col + "_PARSED"].isna(), col + "_PARSED"] = temp.map(count_map)

    # Bird size ordinal
    if "SIZE" in df.columns:
        size_map = {"Small": 1, "Medium": 2, "Large": 3}
        df["SIZE_ORD"] = df["SIZE"].map(size_map)

    # Time of day ordinal
    if "TIME_OF_DAY" in df.columns:
        tod_map = {"Day": 1, "Dawn": 2, "Dusk": 3, "Night": 4}
        df["TIME_OF_DAY_ORD"] = df["TIME_OF_DAY"].map(tod_map)

    return df

X = add_features(X)
X_test = add_features(X_test)

# Align train and test columns
missing_in_test = [col for col in X.columns if col not in X_test.columns]
for col in missing_in_test:
    X_test[col] = np.nan

extra_in_test = [col for col in X_test.columns if col not in X.columns]
X_test = X_test.drop(columns=extra_in_test)

X_test = X_test[X.columns]

# =========================================================
# IDENTIFY COLUMN TYPES
# =========================================================
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["int64", "float64"]).columns.tolist()

# =========================================================
# PREPROCESSING
# =========================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# =========================================================
# MODEL
# =========================================================
if MODEL_TYPE == "xgboost":
    try:
        from xgboost import XGBClassifier
        model = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric="logloss"
        )
    except ImportError:
        raise ImportError(
            "xgboost is not installed. Run: pip install xgboost\n"
            "Or change MODEL_TYPE to 'random_forest'."
        )
else:
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# =========================================================
# TRAIN / VALIDATION SPLIT
# =========================================================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# TRAIN
# =========================================================
clf.fit(X_train, y_train)

# =========================================================
# EVALUATE
# =========================================================
val_preds = clf.predict(X_val)
val_probs = clf.predict_proba(X_val)[:, 1]

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:")
print(classification_report(y_val, val_preds))

try:
    print("Validation ROC-AUC:", roc_auc_score(y_val, val_probs))
except:
    print("ROC-AUC could not be calculated.")

# =========================================================
# TRAIN ON FULL DATA
# =========================================================
clf.fit(X, y)

# =========================================================
# PREDICT TEST
# =========================================================
test_preds = clf.predict(X_test)
test_probs = clf.predict_proba(X_test)[:, 1]

# Save predictions
output = test_df.copy()

# Keep original index column if it exists
if "INDEX_NR" in test_df.columns:
    output = output[["INDEX_NR"]].copy()
else:
    output = pd.DataFrame(index=test_df.index)

output["predicted_damage"] = test_preds
output["predicted_damage_probability"] = test_probs

output.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved predictions to {OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import sklearn 
from sklearn import pipeline
from sklearn import impute
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import FunctionTransformer


train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:

def clean_categorical(X):
    X = X.copy()

    # Example rules:
    # 1. Null in a column means "MissingWasMeaningful"
    if "special_null_col" in X.columns:
        X["special_null_col_was_null"] = X["special_null_col"].isna().astype(int)

    # 2. Null in another column should become "NO"
    if "yes_no_col" in X.columns:
        X["yes_no_col"] = X["yes_no_col"].fillna("NO")

    # 3. Standardize text
    for col in X.columns:
        X[col] = X[col].astype("string").str.strip()

    return X

In [ ]:
# Preprocessing
train_df.drop(columns=['INDEX_NR', 'INCIDENT_DATE', 'AIRPORT_ID', 'AIRPORT', 'LOCATION', 'OPID', 'OPERATOR',
                       'REG', 'FLT', 'AMA', 'AMO', 'EMA', 'EMO', 'PHASE_OF_FLIGHT', 'DISTANCE', 'BIRD_BAND_NUMBER', 'SPECIES', 
                       'OUT_OF_RANGE_SPECIES', 'REMARKS', 'REMAINS_SENT', 'NUM_SEEN', 'ENROUTE_STATE', 'COMMENTS', 'SOURCE', 
                       'PERSON', 'LUPDATE', 'TRANSFER'])
test_df.drop(columns=['INDEX_NR', 'INCIDENT_DATE', 'AIRPORT_ID', 'AIRPORT', 'LOCATION', 'OPID', 'OPERATOR',
                       'REG', 'FLT', 'AMA', 'AMO', 'EMA', 'EMO', 'PHASE_OF_FLIGHT', 'DISTANCE', 'BIRD_BAND_NUMBER', 'SPECIES', 
                       'OUT_OF_RANGE_SPECIES', 'REMARKS', 'REMAINS_SENT', 'NUM_SEEN', 'ENROUTE_STATE', 'COMMENTS', 'SOURCE', 
                       'PERSON', 'LUPDATE', 'TRANSFER'])

numerical_features = ['INCIDENT_MONTH', 'INCIDENT_YEAR', 'LATITUDE', 'LONGITUDE', 'AC_MASS', 'NUM_ENGS', 'ENG_1_POS', 'ENG_2_POS', 
                      'ENG_3_POS', 'ENG_4_POS', 'HEIGHT', 'SPEED', 'REMAINS_COLLECTED']
categorical_features = ['TIME', 'TIME_OF_DAY', 'RUNWAY', 'FAAREGION', 'AIRCRAFT', 'AC_CLASS', 'TYPE_ENG', 'SKY', 'PRECIPITATION', 
                        'SPECIES_ID', 'SIZE', 'WARNED']


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("custom_clean", FunctionTransformer(clean_categorical)),
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

binary_pipeline = Pipeline([
    ("to_binary", FunctionTransformer(lambda X: (~pd.DataFrame(X).isna()).astype(int)))
])

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

full_pipeline.fit(X_train, y_train)
preds = full_pipeline.predict(X_test)